# Generate test manifest

The test manifest contain for each patient / temporality having a GTV, the filename for the gtv and the t1gd.
The GTV is a segmentation based on t1gd, thus t1gd is always available when GTV is.

In [ ]:
import os

manifest = []

sample_wo_gtv = 0
sample_w_t1gd_flair = 0
sample_wo_gtv_w_t1gd_flair = 0

patients = os.listdir('./CFB-GBM')
for patient in patients:
    row = {'patient': patient, 'temporality': None, 't1gd': None, 't2gd': None}
    patient_dir = os.path.join('./CFB-GBM', patient)
    temporalities = os.listdir(patient_dir)
    for temporality in temporalities:
        temporality_dir = os.path.join('./CFB-GBM', patient, temporality)
        files = os.listdir(temporality_dir)
        has_gtv = False
        has_t1gd = False
        has_flair = False
        for file in files:
            modality = file.split('_')[2].split('.')[0]
            print(modality)
            if modality == 't1gd':
                has_t1gd = True
            elif modality == 'gtv':
                has_gtv = True
            elif modality == 'flair':
                has_flair = True

        if not has_gtv:
            sample_wo_gtv += 1
        if has_t1gd and has_flair:
            sample_w_t1gd_flair += 1
        if not has_gtv and has_t1gd and has_flair:
            sample_wo_gtv_w_t1gd_flair += 1

In [ ]:
sample_wo_gtv

In [ ]:
sample_w_t1gd_flair

In [ ]:
sample_wo_gtv_w_t1gd_flair

# Générer le dataset de test

In [ ]:
import os
import shutil
from tqdm import tqdm

source_dir = './CFB-GBM/data'
target_raw = './nnUNet_raw/CFB-GBM_test'

imagestr_dir = os.path.join(target_raw, "imagesTr")
labelstr_dir = os.path.join(target_raw, "labelsTr")
predictionstr_dir = os.path.join(target_raw, "PredictionsTr")
os.makedirs(imagestr_dir, exist_ok=True)
os.makedirs(labelstr_dir, exist_ok=True)
os.makedirs(predictionstr_dir, exist_ok=True)

for patient in tqdm(os.listdir(source_dir)):
    for temporality in os.listdir(os.path.join(source_dir, patient)):
        patient_temporality_id = f'{patient}{temporality[-1]}'
           # Define paths to source files
        files = os.listdir(os.path.join(source_dir, patient, temporality))
        t1gd_filename = f'{patient}_{temporality}_t1gd.nii.gz'
        flair_filename = f'{patient}_{temporality}_flair.nii.gz'
        gtv_filename = f'{patient}_{temporality}_gtv.nii.gz'

        if t1gd_filename in files and flair_filename in files and gtv_filename in files:
            shutil.copy(os.path.join(source_dir, patient, temporality, t1gd_filename), os.path.join(imagestr_dir, f"{patient_temporality_id}_0000.nii.gz"))
            shutil.copy(os.path.join(source_dir, patient, temporality, flair_filename), os.path.join(imagestr_dir, f"{patient_temporality_id}_0001.nii.gz"))
            shutil.copy(os.path.join(source_dir, patient, temporality, gtv_filename), os.path.join(labelstr_dir, f"{patient_temporality_id}.nii.gz"))

# Générer dataset for finetuning

Create a CFB-GBM_finetune_train that contain the sample to finetune the nnUNetv2

Create a CFB-GBM_finetune_test that contain the sample to test the finetuned nnUNetv2

In [ ]:
import os
import shutil
from tqdm import tqdm

source_dir = './nnUNet_raw/CFB-GBM_test'
target_train_raw = './nnUNet_raw/Dataset124_CFB-GBM-finetune-train'
target_test_raw = './nnUNet_raw/Dataset125_CFB-GBM-finetune-test'

imagestr_train_dir = os.path.join(target_train_raw, "imagesTr")
labelstr_train_dir = os.path.join(target_train_raw, "labelsTr")
predictionstr_train_dir = os.path.join(target_train_raw, "PredictionsTr")
os.makedirs(imagestr_train_dir, exist_ok=True)
os.makedirs(labelstr_train_dir, exist_ok=True)
os.makedirs(predictionstr_train_dir, exist_ok=True)

imagestr_test_dir = os.path.join(target_test_raw, "imagesTr")
labelstr_test_dir = os.path.join(target_test_raw, "labelsTr")
predictionstr_test_dir = os.path.join(target_test_raw, "PredictionsTr")
os.makedirs(imagestr_test_dir, exist_ok=True)
os.makedirs(labelstr_test_dir, exist_ok=True)
os.makedirs(predictionstr_test_dir, exist_ok=True)

patients = os.listdir(os.path.join(source_dir, 'labelsTr'))
data_size = len(patients)
shuffled_indices = np.random.permutation(data_size)

split_point = int(data_size * 0.8)

train_indices, test_indices = np.split(shuffled_indices, [split_point])

for train_index in tqdm(train_indices):
    patient = patients[train_index].split('.')[0]
    t1gd_filename_src = f'{patient}_0000.nii.gz'
    flair_filename_src = f'{patient}_0001.nii.gz'
    gtv_filename_src = f'{patient}.nii.gz'

    shutil.copy(os.path.join(source_dir, 'imagesTr', t1gd_filename_src), os.path.join(imagestr_train_dir, f"{patient}_0000.nii.gz"))
    shutil.copy(os.path.join(source_dir, 'imagesTr', flair_filename_src), os.path.join(imagestr_train_dir, f"{patient}_0001.nii.gz"))
    shutil.copy(os.path.join(source_dir, 'labelsTr', gtv_filename_src), os.path.join(labelstr_train_dir, f"{patient}.nii.gz"))

for test_index in tqdm(test_indices):
    patient = patients[test_index].split('.')[0]
    t1gd_filename_src = f'{patient}_0000.nii.gz'
    flair_filename_src = f'{patient}_0001.nii.gz'
    gtv_filename_src = f'{patient}.nii.gz'

    shutil.copy(os.path.join(source_dir, 'imagesTr', t1gd_filename_src), os.path.join(imagestr_test_dir, f"{patient}_0000.nii.gz"))
    shutil.copy(os.path.join(source_dir, 'imagesTr', flair_filename_src), os.path.join(imagestr_test_dir, f"{patient}_0001.nii.gz"))
    shutil.copy(os.path.join(source_dir, 'labelsTr', gtv_filename_src), os.path.join(labelstr_test_dir, f"{patient}.nii.gz"))

# Generate final dir

In [ ]:
os.listdir('./CFB-GBM/data')

In [ ]:
import os
import shutil
from tqdm import tqdm

sample_wo_gtv = 0
sample_w_t1gd_flair = 0
sample_wo_gtv_w_t1gd_flair = 0

source_dir = './CFB-GBM/data'
target_raw = './nnUNet_raw/CFB-GBM_gen'

imagestr_dir = os.path.join(target_raw, "imagesTr")
labelstr_dir = os.path.join(target_raw, "labelsTr")
predictionstr_dir = os.path.join(target_raw, "PredictionsTr")
os.makedirs(imagestr_dir, exist_ok=True)
os.makedirs(labelstr_dir, exist_ok=True)
os.makedirs(predictionstr_dir, exist_ok=True)

patients = os.listdir(source_dir)
for patient in patients:
    print(patient)
    patient_dir = os.path.join(source_dir, patient)
    temporalities = os.listdir(patient_dir)
    for temporality in temporalities:
        temporality_dir = os.path.join(source_dir, patient, temporality)
        files = os.listdir(temporality_dir)

        t1gd_filename = f'{patient}_{temporality}_t1gd.nii.gz'
        flair_filename = f'{patient}_{temporality}_flair.nii.gz'
        gtv_filename = f'{patient}_{temporality}_gtv.nii.gz'
        has_gtv = gtv_filename in files
        has_t1gd = flair_filename in files
        has_flair = t1gd_filename in files
        # for file in files:
        #     modality = file.split('_')[2].split('.')[0]
        #     print(modality)
        #     if modality == 't1gd':
        #         has_t1gd = True
        #     elif modality == 'gtv':
        #         has_gtv = True
        #     elif modality == 'flair':
        #         has_flair = True
        if not has_gtv:
            sample_wo_gtv += 1
        if has_t1gd and has_flair:
            sample_w_t1gd_flair += 1
        if not has_gtv and has_t1gd and has_flair:
            sample_wo_gtv_w_t1gd_flair += 1
        if has_t1gd and has_flair and not has_gtv:
            patient_temporality_id = f'{patient}{temporality[-1]}'
               # Define paths to source files
            files = os.listdir(os.path.join(source_dir, patient, temporality))
            shutil.copy(os.path.join(source_dir, patient, temporality, t1gd_filename), os.path.join(imagestr_dir, f"{patient_temporality_id}_0000.nii.gz"))
            shutil.copy(os.path.join(source_dir, patient, temporality, flair_filename), os.path.join(imagestr_dir, f"{patient_temporality_id}_0001.nii.gz"))

# Organize Gen GTV in a folder

In [ ]:
from pathlib import Path
from tqdm import tqdm
import shutil

gen_gtv_dir = Path('./nnUNet_raw/CFB-GBM_gen/PredictionsTr')
target_dir = Path('./GTV_gen/data')

os.listdir(gen_gtv_dir)

for gtv in tqdm(gen_gtv_dir.glob("*.nii.gz")):
    id_patient_temporality = gtv.name.split('.')[0]
    patient = id_patient_temporality[:-1]
    temporality = id_patient_temporality[-1]
    target_gtv_dir = target_dir / patient / f't{temporality}'

    target_gtv_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(gtv, target_gtv_dir / f'{patient}_t{temporality}_gtv.nii.gz')